## Create a simple nn model, from which create onnx model

Architecture

PyTorch model -> export ONNX -> build TensorRT engine -> run infrence

In [2]:
import torch
import torch.nn as nn


class SimpleNet(nn.Module):

    def __init__(self):
        super().__init__()
    
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3*224*224, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
            return self.net(x)


model = SimpleNet()
model = model.to("cuda")
model.eval()

dummy_input = torch.randn(1, 3, 224, 224).to("cuda")

    

In [3]:
# Export to Onnx
torch.onnx.export(
    model,
    dummy_input,
    "onnx_model/model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=11 )
    

2026-03-20 07:31:32.840181328 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card1/device/vendor"


## Convert onnx to TensorRT engine

In [18]:
!/usr/src/tensorrt/bin/trtexec --onnx=./onnx_model/model.onnx \
--saveEngine=./tensorrt_engine/model.engine \
--fp16 \
--verbose

&&&& RUNNING TensorRT.trtexec [TensorRT v100300] # /usr/src/tensorrt/bin/trtexec --onnx=./onnx_model/model.onnx --saveEngine=./tensorrt_engine/model.engine --fp16 --verbose
[03/20/2026-08:03:55] [I] === Model Options ===
[03/20/2026-08:03:55] [I] Format: ONNX
[03/20/2026-08:03:55] [I] Model: ./onnx_model/model.onnx
[03/20/2026-08:03:55] [I] Output:
[03/20/2026-08:03:55] [I] === Build Options ===
[03/20/2026-08:03:55] [I] Memory Pools: workspace: default, dlaSRAM: default, dlaLocalDRAM: default, dlaGlobalDRAM: default, tacticSharedMem: default
[03/20/2026-08:03:55] [I] avgTiming: 8
[03/20/2026-08:03:55] [I] Precision: FP32+FP16
[03/20/2026-08:03:55] [I] LayerPrecisions: 
[03/20/2026-08:03:55] [I] Layer Device Types: 
[03/20/2026-08:03:55] [I] Calibration: 
[03/20/2026-08:03:55] [I] Refit: Disabled
[03/20/2026-08:03:55] [I] Strip weights: Disabled
[03/20/2026-08:03:55] [I] Version Compatible: Disabled
[03/20/2026-08:03:55] [I] ONNX Plugin InstanceNorm: Disabled
[03/20/2026-08:03:55] [I] 

## Run infrence

In [26]:
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit
import numpy as np

TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

# 1. Load engine
with open("./tensorrt_engine/model.engine", "rb") as f:
    runtime = trt.Runtime(TRT_LOGGER)
    engine = runtime.deserialize_cuda_engine(f.read())

context = engine.create_execution_context()

# 2. Identify Tensors 
inputs = []
outputs = []
for i in range(engine.num_io_tensors):
    name = engine.get_tensor_name(i)
    if engine.get_tensor_mode(name) == trt.TensorIOMode.INPUT:
        inputs.append(name)
    else:
        outputs.append(name)

# Assume single I/O for this specific model
in_name, out_name = inputs[0], outputs[0]

# 1. Convert Dims to a tuple/list
input_shape = tuple(engine.get_tensor_shape(input_name))
output_shape = tuple(engine.get_tensor_shape(output_name))

# 2. Now you can safely use the unpacking operator
h_input = np.random.rand(*input_shape).astype(input_dtype)
h_output = np.empty(output_shape, dtype=output_dtype)

# 5. Allocate Device Memory
d_input = cuda.mem_alloc(h_input.nbytes)
d_output = cuda.mem_alloc(h_output.nbytes)

# 6. Bind addresses to context
context.set_tensor_address(in_name, int(d_input))
context.set_tensor_address(out_name, int(d_output))

# 7. Prepare Data (Match the engine's dtype!)
test_data = np.random.random(input_shape).astype(input_dtype)
np.copyto(h_input, test_data)

# 8. Execution
stream = cuda.Stream()
cuda.memcpy_htod_async(d_input, h_input, stream)
context.execute_async_v3(stream_handle=stream.handle)
cuda.memcpy_dtoh_async(h_output, d_output, stream)

stream.synchronize()

print(f"Output shape: {h_output.shape}")
print(f"Result (first 5): {h_output.flatten()[:5]}")


Output shape: (1, 10)
Result (first 5): [-0.10656738  0.2109375  -0.19006348 -0.08569336 -0.30493164]
